In [1]:
%pip install torch
%pip install --upgrade transformers
%pip install datasets
%pip install peft
%pip install bitsandbytes
%pip install accelerate
%pip install evaluate
%pip install huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated pa

In [2]:
#imports
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
#Global Variables
LORA = True
QUANTIZE = False
CLUSTER = True
CUDA = True
CPU = False
MODEL = "Qwen/Qwen3-4B"
# MODEL = "Qwen/Qwen3-8B"

In [4]:
device = torch.device("cuda" if CUDA and torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [5]:
dataset = load_dataset("glue", "sst2")
train = dataset["train"]
test = dataset["validation"]
train[1]

{'sentence': 'contains no wit , only labored gags ', 'label': 0, 'idx': 1}

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True)

train = train.map(tokenize, batched=True)
test = test.map(tokenize, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb

print("Loading model...")

bnb_config = None
if QUANTIZE:
    print("  Applying 4-bit quantization config...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    quantization_config=bnb_config if QUANTIZE else None,
    device_map="auto" if CUDA else None,
    trust_remote_code=True,
    num_labels=2
)

if LORA:
    print("  Adding LoRA adapters...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=[
            "q_proj", "v_proj", "k_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS"  # <- important
    )

    model = get_peft_model(base_model, lora_config)
else:
    model = base_model

model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

print("Model ready ✅\n")
print(model)

for name, module in model.named_modules():
    print(name)

'NoneType' object has no attribute 'cadam32bit_grad_fp32'
Loading model...


/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Adding LoRA adapters...
Model ready ✅

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Qwen3ForSequenceClassification(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 4096)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vect

In [8]:
from evaluate import load

metric = load("rouge")

def compute_metrics(predictions, references):
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_refs = tokenizer.batch_decode(references, skip_special_tokens=True)
    result = metric.compute(predictions=decoded_preds, references=decoded_refs, use_stemmer=True)
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    return result

In [ ]:
from transformers import TrainingArguments

mode = []
if QUANTIZE and LORA:
    mode.append("quantized_lora")
if QUANTIZE:
    mode.append("quantized")
if LORA:
    mode.append("lora")
if not mode:
    mode.append("fp")  # full precision, no quantization, no LoRA

output_dir = f"./Qwen3-8B-LongBench-finetuned-{'_'.join(mode)}"

training_args = TrainingArguments(
    output_dir=output_dir,
    do_eval=True,
    eval_strategy="steps",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    save_steps=600,
    eval_steps=300,
    logging_steps=300,
    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,
    push_to_hub=False,
    use_cpu=CPU
)

from transformers import Trainer

trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=train,
    eval_dataset=test,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)



/var/folders/lv/pdfd1q0x6qd7nsw0pkqndf700000gn/T/ipykernel_17597/2802113879.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
